In [13]:
res_map = {
    'logs': 'base_simple_scores copy 2.csv',
    'logs2': 'base_simple_scores copy.csv',
    'logs5': 'base_simple_scores.csv'
}

logs_map = {
    'logs': 'Binned Features',
    'logs2': 'Join Key Only',
    'logs5': 'Categorical Features Only'
}

In [19]:
import polars as pl
import os
import json
from pathlib import Path

NOTEBOOK_DIR = Path(os.path.abspath('')).resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parents[1]  # Fast_Data_Discovery

agg_results = []
for k in res_map:
    exp_path = NOTEBOOK_DIR / k
    base_scores = pl.read_csv(PROJECT_ROOT / 'experiments' / 'base_tables' / res_map[k]).rename({'': 'table'})
    aug_scores = pl.read_csv(exp_path / 'simple_results.csv').rename({'_duplicated_0': 'experiment'})
    aug_scores = aug_scores.with_columns(pl.col('experiment').fill_null(pl.col('')), pl.col(['roc_auc', 'f1']).cast(pl.Float64))

    exp_dir = os.listdir(exp_path)
    runtime_res = {}
    for exp in exp_dir:
        if not os.path.isdir(exp_path / exp):
            continue
        comb_log = exp_path / exp / f'{exp}.log'
        with open(comb_log) as f:
            lines = f.readlines()
        for i in range(len(lines)):
            lines[i] = json.loads(lines[i])
        df = pl.from_records(lines, orient='col')
        runtime = df.select(pl.col('runtime').sum()).to_series()[0]
        runtime_res[exp] = runtime
    runtime_df = pl.from_dicts([{'experiment': k, 'runtime': v} for k, v in runtime_res.items()])
    aug_scores = aug_scores.join(runtime_df, left_on='experiment', right_on='experiment', how='left')
    aug_scores = aug_scores.with_columns(
        pl.col('experiment').str.split('_').list.to_struct(upper_bound=3, fields=['lake', 'table', 'algorithm']).struct.unnest()
    )
    aug_scores = aug_scores.join(base_scores, on='table', how='left', suffix='_base')

    aug_scores = aug_scores.with_columns((pl.col('rmse_base').fill_null(0) + pl.col('f1_weighted_base').fill_null(0)).round(3).alias('score_base'))
    aug_scores = aug_scores.with_columns((pl.col('rmse').fill_null(0) + pl.col('f1_weighted').fill_null(0)).round(3).alias('score'))
    aug_scores = aug_scores.with_columns(
        pl.when(pl.col('rmse').is_not_null())
        .then(pl.lit('rmse'))
        .otherwise(pl.lit('f1'))
        .alias('metric')
    )
    # Filter out housing dataset
    aug_scores = aug_scores.filter(pl.col('table').is_in(['housing', 'inspections']).not_()).filter(pl.col('table') != 'elections')
    # Create base rows for each unique (lake, table) combination
    base_rows = (
        aug_scores
        .unique(subset=['lake', 'table'])
        .with_columns([
            pl.lit('base').alias('algorithm'),
            pl.col('score_base').alias('score'),
            pl.col('rmse_base').alias('rmse'),
            pl.col('f1_weighted_base').alias('f1'),
            pl.lit(0.0).alias('runtime'),
            (pl.col('lake') + '_' + pl.col('table') + '_base').alias('experiment')
        ])
        .select(aug_scores.columns)
    )
    aug_scores = pl.concat([aug_scores, base_rows])
    aug_scores = aug_scores.with_columns(
        pl.when(pl.col('table') == 'pageviews')
        .then((pl.col('score')/1e6).round(3))
        .otherwise(pl.col('score'))
        .alias('score')
    )
    aug_scores = aug_scores.with_columns(
        pl.when(
            (pl.col('algorithm') == 'qcr') &
            (pl.col('table').is_in(['arrest', 'food', 'hospital', 'trees']))
        )
        .then(None)
        .otherwise(pl.col('score'))
        .alias('score')
    )
    aug_scores = aug_scores.unique('experiment')
    # Get base displayed score per table (already scaled for pageviews), deduplicated
    base_disp = (
        aug_scores.filter(pl.col('algorithm') == 'base')
        .select('table', pl.col('score').alias('_base_disp'))
        .unique(subset=['table'])
    )

    # Relative improvement per (algorithm, table)
    rel = (
        aug_scores.join(base_disp, on='table')
        .with_columns(
            pl.when(pl.col('metric') == 'rmse')
            .then((pl.col('_base_disp') - pl.col('score')) / pl.col('_base_disp'))
            .otherwise((pl.col('score') - pl.col('_base_disp')) / pl.col('_base_disp'))
            .alias('rel_impr')
        )
    )
    avg_rel = rel.group_by('algorithm').agg(
        (pl.col('rel_impr').filter(pl.col('rel_impr').is_finite()).mean() * 100).round(1).alias('Avg Δ (%)')
    )
    avg_rel = avg_rel.with_columns(pl.lit(logs_map[k]).alias('Setup'))
    aug_scores = aug_scores.join(avg_rel, on='algorithm', how='left')
    agg_results.append(aug_scores)
aug_scores = pl.concat(agg_results).filter(pl.col('algorithm').is_in(['base', 'backward']).not_())

In [17]:
import polars.selectors as cs
aug_scores.select(cs.by_dtype(pl.String))

,experiment,table,f1,roc_auc,lake,algorithm,metric
str,str,str,str,str,str,str,str
"""gittables_vgsales_forward""","""gittables_vgsales_forward""","""vgsales""",null,null,"""gittables""","""forward""","""rmse"""
"""0""","""nyc_trees_kitana""","""trees""",null,null,"""nyc""","""kitana""","""f1"""
"""nyc_trees_autofeat""","""nyc_trees_autofeat""","""trees""",null,null,"""nyc""","""autofeat""","""f1"""
"""0""","""gittables_food_arda""","""food""",null,null,"""gittables""","""arda""","""f1"""
"""nyc_fire_kitana""","""nyc_fire_kitana""","""fire""",null,null,"""nyc""","""kitana""","""rmse"""
…,…,…,…,…,…,…,…
"""0""","""cuk_hospital_kitana""","""hospital""",null,null,"""cuk""","""kitana""","""f1"""
"""0""","""cuk_jobs_arda""","""jobs""",null,null,"""cuk""","""arda""","""rmse"""
"""0""","""nyc_fire_autofeat""","""fire""",null,null,"""nyc""","""autofeat""","""rmse"""


In [20]:
setups_order = ['algorithm', 'Join Key Only', 'Categorical Features Only', 'Binned Features']
algorithm_order = ['base', 'arda', 'autofeat', 'kitana', 'qcr',  'forward', 'backward']

pivoted = (
    aug_scores
        .pivot(
            index='algorithm',
            columns='Setup',
            values='Avg Δ (%)',
            aggregate_function='first'
        )
        .select(setups_order)
        .filter(pl.col('algorithm') != 'base')
        .with_columns(pl.col('algorithm').cast(pl.Enum(algorithm_order)))
        .sort('algorithm')
)
pivoted

/tmp/ipykernel_134523/2990782943.py:6: DeprecationWarning: the argument `columns` for `DataFrame.pivot` is deprecated. It was renamed to `on` in version 1.0.0.
  .pivot(


algorithm,Join Key Only,Categorical Features Only,Binned Features
enum,f64,f64,f64
"""arda""",8.0,5.6,17.3
"""autofeat""",24.7,19.4,9.0
"""kitana""",0.6,0.0,1.3
"""qcr""",21.2,12.1,3.5
"""forward""",36.1,22.6,17.9


In [21]:
import math

TABLES_DIR = PROJECT_ROOT.parent / 'Matryoshka' / 'tables'

setup_display = {
    'Join Key Only': 'Join Key Only',
    'Categorical Features Only': 'Categ. Features',
    'Binned Features': 'Binned Features',
}

algo_display = {
    'arda': 'ARDA', 'autofeat': 'AutoFeat',
    'kitana': 'Kitana', 'qcr': 'QCR',
    'forward': '\\system', 'backward': 'Bwd (\\system)',
}

setup_cols = ['Join Key Only', 'Categorical Features Only', 'Binned Features']

def fmt_val(val):
    if val is None or (isinstance(val, float) and math.isnan(val)):
        return '--'
    sign = '+' if val >= 0 else ''
    return f'{sign}{val:.1f}'

def compute_ranks(vals, higher_is_better=True):
    valid = [(v, i) for i, v in enumerate(vals) if v is not None and not (isinstance(v, float) and math.isnan(v))]
    if not valid:
        return [0] * len(vals)
    sorted_vals = sorted(set(v for v, _ in valid), reverse=higher_is_better)
    best = sorted_vals[0]
    second = sorted_vals[1] if len(sorted_vals) > 1 else None
    ranks = []
    for v in vals:
        if v is None or (isinstance(v, float) and math.isnan(v)):
            ranks.append(0)
        elif v == best:
            ranks.append(1)
        elif second is not None and v == second:
            ranks.append(2)
        else:
            ranks.append(0)
    return ranks

def highlight(s, rank):
    if rank == 1:
        return f'\\color{{red}}\\textbf{{{s}}}'
    elif rank == 2:
        return f'\\underline{{{s}}}'
    return s

# Compute ranks per setup column (higher Avg Δ% is better)
ranks = {}
for col in setup_cols:
    if col in pivoted.columns:
        ranks[col] = compute_ranks(pivoted[col].to_list(), higher_is_better=True)
    else:
        ranks[col] = [0] * len(pivoted)

n_setups = len(setup_cols)
algos = pivoted['algorithm'].to_list()
system_algos = {'forward', 'backward'}

lines = []
lines.append('\\begin{table}[t]')
lines.append('    \\small')
lines.append('    \\centering')
lines.append('    \\setlength\\tabcolsep{4pt}')
lines.append('    \\caption{Average relative improvement (\\%) over the base model across different feature engineering setups. Best results are in \\textbf{bold}, second-best are \\underline{underlined}.}')
lines.append('    \\vspace{-0.3cm}')
lines.append('')
lines.append(f'    \\begin{{tabular}}{{l|{"r" * n_setups}}}')
lines.append('        \\toprule')

# Column headers
header = '        \\textbf{Method}'
for col in setup_cols:
    header += f'\n            & \\textbf{{{setup_display[col]}}}'
header += ' \\\\'
lines.append(header)
lines.append('        \\midrule')

# Data rows
for row_idx, algo in enumerate(algos):
    algo_str = str(algo)
    display_name = algo_display.get(algo_str, algo_str)

    if algo_str in system_algos and (row_idx == 0 or str(algos[row_idx - 1]) not in system_algos):
        lines.append('        \\midrule')

    cells = []
    for col in setup_cols:
        if col in pivoted.columns:
            val = pivoted[col][row_idx]
            val_str = fmt_val(val)
            val_str = highlight(val_str, ranks[col][row_idx])
        else:
            val_str = '--'
        cells.append(val_str)

    lines.append(f'        {display_name}')
    lines.append(f'            & {" & ".join(cells)} \\\\')

lines.append('        \\bottomrule')
lines.append('    \\end{tabular}')
lines.append('')
lines.append('    \\label{table:feature_setups}')
lines.append('\\end{table}')

tex = '\n'.join(lines)
(TABLES_DIR / 'feature_setups.tex').write_text(tex + '\n')
print(tex)

\begin{table}[t]
    \small
    \centering
    \setlength\tabcolsep{4pt}
    \caption{Average relative improvement (\%) over the base model across different feature engineering setups. Best results are in \textbf{bold}, second-best are \underline{underlined}.}
    \vspace{-0.3cm}

    \begin{tabular}{l|rrr}
        \toprule
        \textbf{Method}
            & \textbf{Join Key Only}
            & \textbf{Categ. Features}
            & \textbf{Binned Features} \\
        \midrule
        ARDA
            & +8.0 & +5.6 & \underline{+17.3} \\
        AutoFeat
            & \underline{+24.7} & \underline{+19.4} & +9.0 \\
        Kitana
            & +0.6 & +0.0 & +1.3 \\
        QCR
            & +21.2 & +12.1 & +3.5 \\
        \midrule
        \system
            & \color{red}\textbf{+36.1} & \color{red}\textbf{+22.6} & \color{red}\textbf{+17.9} \\
        \bottomrule
    \end{tabular}

    \label{table:feature_setups}
\end{table}
